# Human grading: run agent on golden cases, read trajectories, grade each criterion

In [1]:
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from agent.llm import LLM
from agent.loop import investigate
from agent.schema import CRITERIA
from agent.tools import LogTools
from agent.trajectory import EventLog, Trajectory
from logstore.db import connect

GOLDEN_PATH = ROOT / "labeling" / "golden_cases.yaml"
GRADES_PATH = ROOT / "labeling" / "human_grades.csv"
CACHE_TAG = "golden"

cases = yaml.safe_load(GOLDEN_PATH.read_text(encoding="utf-8"))
print(f"{len(cases)} golden cases: {[c['id'] for c in cases]}")

10 golden cases: ['c01', 'c02', 'c03', 'c05', 'c06', 'c07', 'c09', 'c10', 'c12', 'c17']


## Run agent on all cases in parallel

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def run_one(case):
    conn = connect()
    tools = LogTools(conn)
    found = tools.logs.find_by_line(case["log_line"])
    assert len(found) == 1, f"{case['id']}: {len(found)} logs match"
    log = found[0]
    tools.visible = log.ts
    request_id = investigate(log, tools, LLM(), EventLog(), cache_tag=CACHE_TAG)
    return case["id"], request_id


with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(run_one, c): c["id"] for c in cases}
    results = {}
    for future in as_completed(futures):
        case_id, request_id = future.result()
        results[case_id] = request_id
        t = Trajectory(request_id)
        print(f"{case_id}: {request_id}  status={t.status}  rounds={t.totals['rounds']}")

print(f"\ndone: {len(results)} investigations")

c05: 6a25f924cb5b  status=ok  rounds=6
c06: c81afc17078d  status=ok  rounds=6
c02: 91061b2c131a  status=ok  rounds=2
c01: 0d4a9a32cb3c  status=ok  rounds=6
c12: 39e2aad29bd7  status=ok  rounds=6
c09: 1477a3af3ad3  status=ok  rounds=6
c03: 82850e27e9c1  status=invalid_verdict  rounds=6
c17: b2408f80507e  status=ok  rounds=4
c07: c4c95ae99316  status=model_error  rounds=6


## Read each trajectory and grade it, one at a time
Saves after each case so you can stop and resume.

In [ ]:
# load existing grades if any
if GRADES_PATH.exists():
    grades_df = pd.read_csv(GRADES_PATH)
    graded = set(grades_df.case_id)
    rows = grades_df.to_dict("records")
    print(f"already graded: {sorted(graded)}")
else:
    graded = set()
    rows = []

for case_id in sorted(results):
    if case_id in graded:
        continue
    t = Trajectory(results[case_id])
    print("=" * 100)
    print(f"CASE {case_id} — request {results[case_id]}")
    print("=" * 100)
    print(t.text())
    print()
    row = {"case_id": case_id, "request_id": results[case_id]}
    for criterion in CRITERIA:
        while True:
            answer = input(f"  {criterion} (pass/fail): ").strip().lower()
            if answer in ("pass", "fail"):
                row[criterion] = answer
                break
            print("    type 'pass' or 'fail'")
    rows.append(row)
    # save after each case
    pd.DataFrame(rows).to_csv(GRADES_PATH, index=False)
    print(f"  saved ({len(rows)} rows so far)\n")